# Roteiro de Análise de Dados - Fórmula 1

Este notebook implementa as etapas sugeridas no roteiro de análise, cobrindo desde o carregamento dos dados até a aplicação de modelos de Machine Learning (Clusterização, Classificação e Regressão).

## 1. Configuração e Importação de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score

# Configurações de visualização
sns.set_palette("husl")
sns.set_style("whitegrid")
%matplotlib inline

print("Bibliotecas importadas com sucesso!")

## 2. Carregamento e Preparação dos Dados

Vamos carregar os arquivos principais: `results.csv`, `races.csv`, `drivers.csv`, `constructors.csv` e `status.csv`.
Importante: O dataset usa `\N` para representar valores nulos.

In [ ]:
base_path = 'data/'

try:
    results = pd.read_csv(base_path + 'results.csv', na_values='\\N')
    races = pd.read_csv(base_path + 'races.csv', na_values='\\N')
    drivers = pd.read_csv(base_path + 'drivers.csv', na_values='\\N')
    constructors = pd.read_csv(base_path + 'constructors.csv', na_values='\\N')
    status = pd.read_csv(base_path + 'status.csv', na_values='\\N')
    print("Arquivos CSV carregados com sucesso.")
except FileNotFoundError as e:
    print(f"Erro ao carregar arquivos: {e}")

In [ ]:
# Merge dos Datasets para formar um DataFrame consolidado
# Juntando Resultados com Corridas (para pegar ano e circuito)
df = results.merge(races[['raceId', 'year', 'name', 'date', 'circuitId']], on='raceId', how='left')

# Juntando com Pilotos (nome, nacionalidade)
df = df.merge(drivers[['driverId', 'driverRef', 'nationality', 'dob']], on='driverId', how='left')

# Juntando com Construtores (equipes)
df = df.merge(constructors[['constructorId', 'constructorRef', 'nationality']], on='constructorId', how='left', suffixes=('_driver', '_team'))

# Juntando com Status (causa do fim da corrida)
df = df.merge(status[['statusId', 'status']], on='statusId', how='left')

# Limpeza Rápida
# Convertendo posição para numérico (substituindo o que não for numérico por NaN, embora ja tenhamos tratado \N)
df['positionOrder'] = pd.to_numeric(df['positionOrder'], errors='coerce')
df['grid'] = pd.to_numeric(df['grid'], errors='coerce')
df['points'] = pd.to_numeric(df['points'], errors='coerce')

print(f"Dataset consolidado criado. Formato: {df.shape}")
df.head()

## 3. Análise Exploratória de Dados (EDA)

In [ ]:
# Top 10 Pilotos com mais vitórias na base de dados
wins = df[df['positionOrder'] == 1].groupby('driverRef')['year'].count().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=wins.values, y=wins.index, palette='viridis')
plt.title('Top 10 Pilotos por Número de Vitórias')
plt.xlabel('Vitórias')
plt.show()

In [ ]:
# Correlação entre Posição de Largada (Grid) e Posição Final
# Filtrando apenas corridas completadas (sem abandonos)
df_finished = df[df['statusId'] == 1] # statusId 1 geralmente é 'Finished'

plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_finished, x='grid', y='positionOrder', alpha=0.1)
plt.title('Correlação: Grid de Largada vs Posição Final')
plt.xlabel('Posição de Largada')
plt.ylabel('Posição Final')
plt.show()

corr = df_finished[['grid', 'positionOrder']].corr().iloc[0,1]
print(f"Correlação entre Largada e Chegada: {corr:.2f}")

## 4. Clusterização (Modelagem Não-Supervisionada)
**Objetivo:** Agrupar pilotos com base em suas estatísticas médias (Posição média, Pontos médios).

In [ ]:
# Criando métricas por piloto
driver_stats = df.groupby('driverRef').agg({
    'positionOrder': 'mean',
    'points': 'mean',
    'raceId': 'count'
}).reset_index()

# Filtrando pilotos com poucas corridas (> 20 corridas) para evitar ruído
driver_stats = driver_stats[driver_stats['raceId'] > 20]

# Preparando dados para o modelo
X_cluster = driver_stats[['positionOrder', 'points']]

# Normalização (Importante para K-Means)
scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

# Aplicando K-Means
kmeans = KMeans(n_clusters=4, random_state=42)
driver_stats['cluster'] = kmeans.fit_predict(X_cluster_scaled)

# Visualizando os Clusters
plt.figure(figsize=(10, 6))
sns.scatterplot(data=driver_stats, x='positionOrder', y='points', hue='cluster', palette='deep', s=100)
plt.title('Clusters de Pilotos: Posição Média vs Pontos Médios')
plt.xlabel('Posição Média (Menor é melhor)')
plt.ylabel('Pontos Médios (Maior é melhor)')
plt.show()

## 5. Classificação (Modelagem Supervisionada)
**Objetivo:** Prever se um piloto terminará no **Pódio (Top 3)**.

In [ ]:
# Criando a variável alvo (Target)
df['is_podium'] = df['positionOrder'].apply(lambda x: 1 if x <= 3 else 0)

# Selecionando Features
# Vamos usar: Grid, Ano (para capturar tendências temporais) e Equipe
le_team = LabelEncoder()
df['team_code'] = le_team.fit_transform(df['constructorRef'].astype(str))

features = ['grid', 'team_code', 'year']

# Limpando NaNs nas features
df_model = df.dropna(subset=features)

X = df_model[features]
y = df_model['is_podium']

# Divisão Treino/Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Modelo Random Forest
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Avaliação
y_pred = clf.predict(X_test)
print("Acurácia:", accuracy_score(y_test, y_pred))
print("\nRelatório de Classificação:\n", classification_report(y_test, y_pred))

## 6. Regressão (Modelagem Supervisionada)
**Objetivo:** Prever a quantidade de **pontos** obtidos na corrida.

In [ ]:
# Target: points
# Usando as mesmas features da classificação
y_reg = df_model['points']

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.3, random_state=42)

# Modelo Regressão Linear
reg = LinearRegression()
reg.fit(X_train_reg, y_train_reg)

# Predição
y_pred_reg = reg.predict(X_test_reg)

# Avaliação
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2 = r2_score(y_test_reg, y_pred_reg)

print(f"RMSE: {rmse:.2f}")
print(f"R2 Score: {r2:.2f}")

# Plot Real vs Previsto
plt.figure(figsize=(8, 6))
plt.scatter(y_test_reg, y_pred_reg, alpha=0.3)
plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 'r--', lw=2)
plt.xlabel('Pontos Reais')
plt.ylabel('Pontos Previstos')
plt.title('Regressão: Pontos Reais vs Previstos')
plt.show()